# Import packages

In [1]:
from bs4 import BeautifulSoup as bs     # parsing
from urllib.request import urlopen      # scrapping
import pandas as pd                     # exporting
from tenacity import retry, stop_after_attempt
import http.client

In [2]:
from io import StringIO
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.pdfpage import PDFPage
import os
import logging
from tqdm import tqdm
import csv
import re

In [3]:
address_1 = re.compile('(?:ONE|\d{1,4})?[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)
address_2 = re.compile('(?:ONE|\d{1,4})[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)

city = re.compile('[\w\s]{1,20}[,] \w{1,}?[\d\W]{1,11} [\w\s]{1,3}')
date = re.compile('(?:(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?\s+(?:of\s+)?(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)|(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)\s+(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?)(?:\,)?\s*(?:\d{4})?|[0-3]?\d[-\./][0-3]?\d[-\./]\d{2,4}', re.IGNORECASE)
number = re.compile('\d{1,20}')
title = ['PRESIDENT', 'VICE PRESIDENT', 'VICE PRESIDENT, TAX', 'TREASURER', 'ASSISTANT TREASURER', 'ASST. TREASURER', 'ASSISTANT CLERK', 'CLERK',
         'SECRETARY', 'ASSISTANT SECRETARY', 'CORPORATE SECRETARY', 'GENERAL SECRETARY', 'DIRECTOR', 'SECRETARY & DIRECTOR', 'MEDICAL DIRECTOR',
         'ASST TREAS/SECY', 'EDUCATION DIRECTOR', 'CO-WARDEN', 'VICE PRESIDENT FOOTBALL', 'VICE PRESIDENT CHEER', 'REGISTRAR', 'PUBLIC RELATIONS',
         'SENIOR PASTOR', 'ASSISTANT PASTOR', 'CHURCH LEADER', 'COLLECTOR', 'MODERATOR', 'CHAIR', 'GENERAL MEMBER', 'CEO', 'CFO',
         'GENERAL EVANGELIST', 'EXECUTIVE MINISTER', 'CORP CONTROLLER', 'CORPORATE CONTROLLER', 'OFFICER', 'CHAIRMAN & CEO', 'MANAGER',
         'PRESIDENT & CHIEF OPERATING', 'OPERATIONS', 'EDUCATION', 'CHAIRMAN', 'ICE DIRECTOR', 'CHIEF FINANCIAL OFFICER', 'DIRECTOR OF FINANCE',
         'CHIEF EXECUTIVE OFFICER', 'CHIEF OPERATING OFFICER', 'CHIEF COMPLIANCE OFFICER', 'IMMEDIATE PAST CHAIR', 'CHAIR-ELECT',
         'DIRECTOR-CHAIRMAN', 'CFO-TREASURER', 'EVP-COO', 'SENIOR MINISTER', 'MISSIONARY', 'ARCHIVIST', 'SIGNATORY', 'REAL PROPERTY']

# Util functions

In [4]:
def process_pdf(rsrcmgr, device, fp, pagenos=None, maxpages=0, password='', caching=True, check_extractable=True):
    interpreter = PDFPageInterpreter(rsrcmgr, device)
    for page in PDFPage.get_pages(fp, pagenos, maxpages=maxpages, password=password,
                                  caching=caching, check_extractable=check_extractable):
        interpreter.process_page(page)
    return

In [5]:
def read_pdf(pdf):
    rsrcmgr = PDFResourceManager()
    retstr = StringIO()
    laparams = LAParams()
    device = TextConverter(rsrcmgr, retstr, laparams=laparams)
    process_pdf(rsrcmgr, device, pdf)
    device.close()
    content = retstr.getvalue()
    retstr.close()
    lines = str(content).split("\n")
    return lines

In [12]:
def handle(rst, phase):
    '''
    param rst: str, the object representing the info extracted from the pdf, usually the result from function read_pdf
    param phase: int, the phase protocol, to signal which regex compiler should be in use
    '''
#     print(rst)
    idnum = ''
    exactName = ''
    jurisdiction = ''
    registeredName = ''
    registeredAddress = ''
    principalAddress = ''
    directors_info = []
    business = ''
    address = None

    INFO = []

    if phase == 1:
        address = address_1
    elif phase == 2:
        address = address_2
    
    for i in range(len(rst)):
        if 'Identification Number' in rst[i]:
            idnum = rst[i].split(': ')[1].strip()
        
        if 'Exact name' in rst[i] or 'exact name' in rst[i]:
            exactName = rst[i].split(': ')[1].lstrip()
                
            count = i + 1
            tmp = rst[count]
            while 'Jurisdiction of Incorporation' not in tmp and 'registered office' not in tmp and \
                  'principal office' not in tmp:
                exactName += tmp.strip()
                count += 1
                tmp = rst[count]
            
            exactName = exactName.split('2.')[0].strip()
            
        if 'Jurisdiction of Incorporation' in rst[i]:
            state = ''
            country = ''
            count = i
            tmp = rst[count]
            while 'registered office' not in tmp and 'principal office' not in tmp:
                if 'State' in tmp:
                    state = tmp.split('State: ')[1][0:2]
                if 'Country' in tmp:
                    country = tmp.split('Country: ')[1].strip()
                count += 1
                tmp = rst[count]

            if len(country) != 0:
                jurisdiction = state + ', ' + country
            else:
                jurisdiction = state
            
        if 'corporation registered office' in rst[i]:
            registeredStreetList = []
            registeredStreet = ''
            registeredCity = ''
            registeredState = ''
            registeredZip = ''
            registeredCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'principal office' not in tmp and 'directors' not in tmp:
                if (address.search(tmp) != None) and (tmp.isupper()):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        registeredStreetList.append(tmp.strip())
                    else:
                        registeredStreetList.append(tmp.split(',')[0].strip())
                elif registeredName == '':
                    if tmp.isupper():
                        registeredName = tmp.strip()
                elif tmp.isupper() and tmp.replace(' ', '').replace('.', '').isalpha():
                    registeredCity = tmp.strip()
                    
                if 'State' in tmp:
                    try:
                        registeredState = tmp.split('State: ')[1].strip()
                    except:
                        registeredState = ''
                if 'Zip' in tmp:
                    try:
                        registeredZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        registeredZip = ''
                if 'Country' in tmp:
                    registeredCountry = tmp.split('Country: ')[1].strip()
                    
                count += 1
                tmp = rst[count]
            
            for i in range(len(registeredStreetList)):
                registeredStreet += registeredStreetList[i]
                registeredStreet += ', '
            
            registeredAddress = registeredStreet + registeredCity + ', ' + registeredState + \
                                ' ' + registeredZip + ' ' + registeredCountry
            
        if 'principal office' in rst[i] and 'location' in rst[i].lower():
            principalStreetList = []
            principalStreet = ''
            principalCity = ''
            principalState = ''
            principalZip = ''
            principalCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'directors' not in tmp:
                if 'Commonwealth' in tmp:
                   break
                if tmp.isupper() and '3.' not in tmp:
                    if principalStreet == '':
                        if ',' in tmp and tmp.strip().split(',')[1] != '':
                            principalStreet = tmp.strip()
                        else:
                            principalStreet = tmp.split(',')[0].strip()
                if 'CITY' in tmp.upper():
                    principalCity = rst[count+3]
                
                if 'State' in tmp:
                    try:
                        principalState = tmp.split(': ')[1].strip()
                    except:
                        principalState = ''
                if 'Zip' in tmp:
                    try:
                        principalZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        principalZip = ''
                if 'Country' in tmp:
                    principalCountry = tmp.split(': ')[1].strip()
                
                count += 1
                # print(count)
                # print('City', principalCity)
                # print('Street', principalStreet)
                # print('State', principalState)
                # print('Zip', principalZip)
                # print('Country', principalCountry)
                tmp = rst[count]
            
            # for i in range(len(principalStreetList)):
            #     principalStreet += principalStreetList[i]
            #     principalStreet += ', '
            
            principalAddress = principalStreet + ' ' + principalState + \
                               ' ' + principalZip + ' ' + principalCountry
            
        if 'Title' in rst[i]:
            directorsTitle = []
            directorsName = []
            directorsStreet = []
            directorsCity = []
            count = i + 1
            tmp = rst[count]
            while 'Briefly describe' not in tmp and 'cemetery corporation' not in tmp and 'undersigned' not in tmp:
                if 'Additional matter' in tmp or 'Resident Agent' in tmp:
                    break
                # sometimes the title name varies, we can just check if it contains any one in the title list
                if tmp.replace('\xa0', '').strip() in title or \
                bool([t for t in title if (t in tmp.replace('\xa0', '').strip())]): 
                    directorsTitle.append(tmp.strip())
                elif ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                    directorsCity.append(tmp.strip())
                
                # usually the column of name is on the left of the address, so this one step should go first
                elif (tmp.isupper()) and (len(tmp.split()) > 1) and \
                     (re.sub(r'[^\w\s]','',tmp).replace('\xa0', '').replace(' ', '').isalpha()):
                    if address.search(tmp) == None:
                        directorsName.append(tmp.strip())
                    elif ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsName.append(tmp.strip())
                    else:
                        directorsName.append(tmp.split(',')[0].strip())
                        
                elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp) or (tmp.strip() == 'SAME'):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                    
                count += 1
                print(count)
                print(tmp)
                tmp = rst[count]
            
            directorsName = [' '.join(x) for x in zip(directorsName[0::2], directorsName[1::2])] if len(directorsName) >= len(directorsCity) else directorsName
            directorsCity = [', '.join(x) for x in zip(directorsCity[0::2], directorsCity[1::2])]
            if len(directorsCity) == 0:
                while 'Check' not in tmp:
                    if ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                        directorsCity.append(tmp.strip())
                    elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp):
                        directorsStreet.append(tmp.split(',')[0].strip())
                        
                    count += 1
                    tmp = rst[count]
                
            
            directors_info = [[] for _ in range(len(directorsTitle))]

            if len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet) and \
                len(directorsStreet) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                             directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsStreet):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                             directorsStreet[i])
            elif len(directorsTitle) == len(directorsName) and len(directorsName) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + directorsCity[i])
            elif len(directorsStreet) == len(directorsName) and len(directorsName) == len(directorsCity):
                for i in range(len(directorsStreet)):
                    directors_info[i].append(directorsName[i] + ', ' + \
                                             directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsName):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i])
            elif len(directorsTitle) == len(directorsCity):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsCity[i])
            elif len(directorsTitle) == len(directorsStreet):
                for i in range(len(directorsTitle)):
                    directors_info[i].append(directorsTitle[i] + ', ' + directorsStreet[i])
            elif len(directorsCity) == len(directorsName):
                for i in range(len(directorsName)):
                    directors_info[i].append(directorsName[i] + ', ' + directorsCity[i])
            elif len(directorsCity) == len(directorsStreet):
                for i in range(len(directorsCity)):
                    directors_info[i].append(directorsStreet[i] + ', ' + directorsCity[i])
            elif len(directorsName) == len(directorsStreet):
                for i in range(len(directorsName)):
                    directors_info[i].append(directorsName[i] + ', ' + directorsStreet[i])
            INFO.extend(directors_info)
                
        if 'The general character' in rst[i] or 'Briefly describe' in rst[i]:
            count = i + 1
            tmp = rst[count]
            while 'cid' in tmp or tmp == ' ' or tmp == '' or 'service, the service to be rendered:' in tmp or \
                address.search(tmp) != None or city.search(tmp) != None or tmp.isupper() == False:
                count += 1
                tmp = rst[count]
                break
            business = tmp.strip()
            
    try:
        id = format(int(idnum), '09d') if len(idnum) <= 9  and len(idnum) else None
    except:
        id = idnum
                    
    rec = {

        'ID Number': id, # fixed length = 9

        'Business Name': exactName,

        'Jurisdiction': jurisdiction,
        
        'Registered Office Name': registeredName,
        
        'Registered Office Address': registeredAddress,
        
        'Principal Office Address': principalAddress,
        
        'Directors': directors_info,
        
        'Business': business
        
        }
    
    return rec

In [15]:
path = '.\PDF_data\PDF3'
dirs = os.listdir(path)
# business = {}
results = [] 
for file in tqdm(dirs):
    pdf = path + '/' + file
    with open(pdf, 'rb') as f:
        try:
            rst = read_pdf(f)
        except:
            continue
        try:
            info = handle(rst, 2)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = format(int(file[:-4]), '09d')
            results.append(info)
        except IndexError:
            # print('list index out of range')
            continue
        except ValueError:
            info = handle(rst, 2)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = file[:-4]
            results.append(info)
            continue

  0%|                                                                                | 1/10346 [00:00<43:28,  3.97it/s]

78

79
PRESIDENT  
80

81
Individual Name
82
First, Middle, Last, Suffix 
83
STEVE IANNACCONE   
84

85
TREASURER  
86

87
THOMAS MCGRATH   
88

89
CLERK  
90

91
JEFF ROSS   
92

93
CO-PRESIDENT  
94

95
CLARIE NAUGHTON   
96

97
DIRECTOR  
98

99
STEVE DRISCOLL   
100

101
DIRECTOR  
102

103
THOMAS BARBERA   
104

105
        26 ROCK AVENUE #2
106

107
SWAMPSCOTT, MA 01907 USA  
108

109
        72 HOLLIS STREET
110

111
PEPPERELL, MA 01463 USA  
112

113
        20 PARK STREET
114
BOSTON, MA 02116 USA  
115
        4 EVERETT LANE
116
FOXBORO, MA 02035 USA  
117
        47 SHORES EDGE
118

119
PEMBROKE, MA 02359 USA  
120
        24 MORTON STREET
121
WALTHAM, MA 02453 USA  
122

123
2010  
124

125
2010  
126

127
2010  
128

129
2010  
130

131
2010  
132

133
2010  
134

135
2010  
136

137
 
138
 
139
 
140
 
141
 
142
 
143
 
144
dates shown. IN WITNESS WHEREOF AND UNDER PENALTIES OF PERJURY, I hereto sign my 
145
name on this 7 Day of September, 2010. 
146

147
© 2001 - 2010 C

  0%|                                                                                | 3/10346 [00:00<36:32,  4.72it/s]

  0%|                                                                                | 6/10346 [00:01<39:27,  4.37it/s]

89

90
Individual Name
91
First, Middle, Last, Suffix 
92

93
MANAGER  
94

95
BANCBOSTON INVESTMENTS, INC.   
96

97
Address (no PO Box)
98

99
Address, City or Town, State, Zip Code 
100

101
  100 FEDERAL ST.
102

103
BOSTON, MA 02110-0000 USA  
104

105
7. The name and business address of the person in addition to the manager, who is authorized to
106
execute documents to be filed with the Corporations Division, and at least one person shall be
107
named if there are no managers.  
108

109
Title 
110

111
Individual Name
112
First, Middle, Last, Suffix 
113

114
SOC SIGNATORY  
115

116
GREG S MROZ   
117

118
SOC SIGNATORY  
119

120
NINA TAI   
121

122
Address (no PO Box)
123

124
Address, City or Town, State, Zip Code 
125
  401 N TRYON ST; NC1-021-02-20
126

127
CHARLOTTE, NC 28255 USA  
128

129
  231 S LASALLE ST; IL1-231-12-34
130

131
CHICAGO, IL 60604 USA  
132

133
8. The name and business address of the person(s) authorized to execute, acknowledge, deliver and
134
rec

  0%|                                                                                | 7/10346 [00:01<40:01,  4.31it/s]

... (178 outputs left out)


88

89
CAMBRIDGE, MA 02138 USA  
90

91
6. The latest date on which the limited partnership is to dissolve:  12/31/2051 
92



  1%|▉                                                                             | 132/10346 [00:27<20:16,  8.39it/s]

134
First Name: MATTHEW
135
Residential Address: 2 KNOB HILL  
136
City: GT BARRINGTON
137
Country: USA
138

139
Title: CLERK
140
First Name: ANN
141
Residential Address: 2 KNOB HILL  
142
City: GT. BARRINGTON
143
Country: USA
144

145
Title: DIRECTOR
146
First Name: MATTHEW
147
Residential Address: 2 KNOB HILL  
148
City: GT. BARRINGTON
149
Country: USA
150

151
Title: DIRECTOR
152
First Name: ANN
153
Residential Address: 2 KNOB HILL  
154
City: GT. BARRINGTON
155
Country: USA
156

157
Title: TREASURER
158
First Name: ANN
159
Residential Address: 2 KNOB HILL  
160
City: GT. BARRINGTON
161
Country: USA
162

163
Middle Name: 
164

165
Expiration of Term: none
166
Last Name: SAGARIN
167

168
State: MA
169

170
Zip: 01230
171

172
Middle Name: 
173

174
Expiration of Term: none
175
Last Name: SAGARIN
176


KeyboardInterrupt: 

In [14]:
resultsDF

,ID Number,Business Name,Jurisdiction,Registered Office Name,Registered Office Address,Principal Office Address,Directors,Business
0,001064093,"A ST. AUBIN COMPANIES, INC.",MA,RICHARD ST. AUBIN,"460 WHISTLEBERRY DR, MARSTONS MILLS, MA 02648 USA",,"[[PRESIDENT, 186 BENNETT ROAD], [TREASURER, 18...",
1,000142040,"IMAGEONE INDUSTRIES,INC.","PA, USA","VCORP AGENT SERVICES, INC.","44 SCHOOL STREET, SUITE 325, BOSTON, MA 02108 USA",,"[[PRESIDENT, 677 DUNKSFERRY ROAD], [TREASURER,...",GENERAL CONTRACTING
2,001147662,COPICUT CORP.,MA,ELIZABETH ANNE CASNER,"189 WELLS AVE., SUITE 200, NEWTON, MA",,"[[PRESIDENT, 19 ROCKLAND ST.], [TREASURER, 19 ...",REAL ESTATE
3,001088977,"PIACENTINI CONSTRUCTION, CORP.",MA,ADRIANO GONTIGIO,"255 THACHER ST., MILTON, MA 02186 USA",,"[[PRESIDENT, 255 THACHER ST], [TREASURER, 255 ...",HOME IMPROVEMENT AND REMODELING
4,900679731,"A.S. ADVISORY, LLC",,,,54 RUSSELL STREET MA 02129 USA,"[[REAL PROPERTY, 54 RUSSELL STREET], [REAL PRO...","CONSULTING PRACTICE FOCUSED ON: VALUATION, COM..."
5,001397225,AKB HARDWOOD FLOOR INC,MA,ARNOBIO DE PAULA SOARES,"45 EDEN ST APT 3, FRAMINGHAM, MA 01702 USA",,"[[PRESIDENT, 45 EDEN ST APT 3], [TREASURER, 45...",FLOORING INSTALLATION
6,041222630,"ALBERT CULVER COMPANY, THE",MA,WILLIAM F. SPENCE,"83 EAST WATER STREET, ROCKLAND, MA 02370 USA",,"[[PRESIDENT, 83 EAST WATER STREET], [TREASURER...","SALES AND SERVICE, FUEL OIL"
7,000186980,GREENTREE MARKETING INC.,MA,LEE K JENSEN,"FRAMINGHAM, MA 01701 USA",,"[[PRESIDENT, 567 HEMENWAY ST.], [TREASURER, 56...",PROMOTIONAL PRODUCTS
8,271434866,"BJSB, INC.",MA,BENJAMIN BAHNAN,"344 PLEASANT STREET, WORCESTER, MA 01609 USA",,"[[PRESIDENT, 57 WRENTHAM ROAD], [PRESIDENT, 57...",WHOLESALE AND RETAIL OF FOOD AND FOOD SERVICE
9,200169885,"AGRIVIDA, INC.","DE, USA",R. MICHAEL RAAB,"200 BOSTON AVE., STE. 3100, MEDFORD, MA 02155 USA",,"[[], [], [], [], []]",GROWING CORN EXPRESSING ENZYMES FOR ANIMAL FEED


In [11]:
resultsDF = pd.DataFrame(results)
resultsDF.to_csv('~/Desktop/sec_results.csv')

In [102]:
path = './../Desktop/PDF_extracted' 
for file in tqdm(resultsDF['ID Number']):
    pdf = path + '/' + file + '.pdf'
    if os.path.exists(pdf):
        shutil.move(pdf, './../Desktop/PDF_extracted' + '/' + file + '.pdf')


100%|███████████████████████████████████████████████████████████████████████████| 3977/3977 [00:00<00:00, 21270.44it/s]


In [104]:
result = pd.read_csv('~/Desktop/results/sec_results1.csv')
result = pd.concat([result, resultsDF], ignore_index=0)
result.to_csv('~/Desktop/sec_results1.csv')